# Battery Cell Design — RL Training on Google Colab

Train an LLM to design lithium-ion battery cells using real PyBaMM electrochemistry as the reward signal.

## What you get from this notebook

| Episodes | What you see |
|----------|-------------|
| 0 | Zero-shot baseline captured (before RL) |
| 100 | Loss trending down, early reward improvement visible |
| 500 | Meaningful learning curve, pass rate starting to rise |
| 2000+ | Model consistently passing easy tasks, some medium |
| 10K+ | Strong results across difficulty tiers |

**500 episodes on a free T4 (~40 min) will show a real learning curve.**
It will not produce a fully trained model — but it proves the loop works and gives you a chart.

## Memory strategy (T4, 16 GB VRAM)

| Technique | VRAM saved |
|-----------|------------|
| 4-bit NF4 quantisation (Unsloth) | ~8 GB |
| LoRA r=16 (train 40M not 7B params) | ~12 GB optimizer |
| Unsloth gradient checkpointing | ~30% activations |
| Dataset-mode rewards (Parquet KDTree) | 0 GPU — reward in 1 ms not 3 s |

**Result:** model fits in ~7 GB, leaving 9 GB for activations + batch.

## Runtime
**GPU required.** Runtime → Change runtime type → T4 GPU.

## 0. Configuration — all tunable parameters here

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME  = 'unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit'
# Smaller alternatives (less VRAM, faster iteration):
# MODEL_NAME = 'unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit'  # ~2 GB
# MODEL_NAME = 'unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit' # ~1 GB
MAX_SEQ_LEN = 1024   # prompt ~400 tokens + output ~300 tokens = ~700; 1024 is safe
LORA_RANK   = 16

# ── Training ──────────────────────────────────────────────────────────────────
N_EPISODES       = 500    # 500 = learning curve demo; 2000+ = real training
LEARNING_RATE    = 2e-4
ACCUM_STEPS      = 4      # gradient accumulation: effective batch = 4
BASELINE_WINDOW  = 30     # rolling mean window for REINFORCE advantage baseline
TEMP_START       = 0.9    # generation temperature at episode 1
TEMP_END         = 0.3    # generation temperature at episode N_EPISODES
CHECKPOINT_EVERY = 100    # save adapter + history every N episodes
VIZ_EVERY        = 50     # redraw dashboard every N episodes
MAX_GRAD_NORM    = 1.0

# ── Curriculum ────────────────────────────────────────────────────────────────
# Easy task probability anneals from EASY_START → EASY_END over first half
EASY_START = 0.60
EASY_END   = 0.20

# ── Dataset ───────────────────────────────────────────────────────────────────
N_DATASET_SAMPLES = 2000   # 2000 = good coverage; 500 = quick demo
DATASET_PATH      = 'data/battery_dataset.parquet'

# ── Checkpointing ─────────────────────────────────────────────────────────────
# Set USE_DRIVE=True to persist checkpoints across Colab sessions
USE_DRIVE         = False
DRIVE_DIR         = '/content/drive/MyDrive/prodata_battery'
CHECKPOINT_DIR    = DRIVE_DIR if USE_DRIVE else 'checkpoints/battery'

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42

print('Config loaded.')
print(f'  Model:      {MODEL_NAME}')
print(f'  Max seq len:{MAX_SEQ_LEN}')
print(f'  Episodes:   {N_EPISODES}')
print(f'  Dataset:    {N_DATASET_SAMPLES} samples')
print(f'  Checkpoint: every {CHECKPOINT_EVERY} episodes → {CHECKPOINT_DIR}')

## 1. Install

In [ ]:
# Step 1 of 2: Unsloth (must install before transformers)
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

# Step 2 of 2: Prodata gym from GitHub + all battery dependencies
# (prodata is not on PyPI — install directly from the repo)
!pip install "git+https://github.com/prodata-ai/ProdataGym.git" -q
!pip install pybamm pandas scipy matplotlib pyarrow -q

# TRL for GRPO (Appendix)
!pip install "trl>=0.12" datasets -q

print("Install complete. Verifying key packages...")
import importlib
for pkg in ["pybamm", "prodata", "unsloth", "trl", "gymnasium", "pyarrow"]:
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, "__version__", "ok")
        print(f"  {pkg}: {ver}")
    except ImportError:
        print(f"  {pkg}: MISSING — re-run this cell")

In [ ]:
import pybamm, torch, gymnasium, prodata
print(f'PyBaMM:    {pybamm.__version__}')
print(f'PyTorch:   {torch.__version__}')
print(f'CUDA:      {torch.cuda.is_available()} | device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
if not torch.cuda.is_available():
    raise RuntimeError('GPU not found. Runtime → Change runtime type → T4 GPU')

## 2. Google Drive (optional — persists checkpoints across sessions)

In [ ]:
import os
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
    print(f'Drive mounted. Checkpoints → {DRIVE_DIR}')
else:
    print('Drive not used. Checkpoints saved locally (lost on session end).')
    print('Set USE_DRIVE=True in the Config cell to persist across sessions.')

Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
Path('data').mkdir(exist_ok=True)

## 3. Pre-compute dataset

Runs real PyBaMM simulations once. During training, reward lookups cost ~1 ms (KDTree) instead of ~3 s (live PyBaMM).

| Samples | Time (Colab CPU) | Coverage |
|---------|-----------------|----------|
| 500 | ~5 min | thin, good for quick demo |
| 2000 | ~20 min | recommended default |
| 10000 | ~90 min | serious training |

In [ ]:
import pandas as pd
import numpy as np

dataset_path = Path(DATASET_PATH)

if dataset_path.exists():
    df = pd.read_parquet(dataset_path)
    print(f'Dataset exists: {len(df):,} rows — skipping generation.')
    print(df[['chemistry','energy_density_whkg','cycle_life_80pct','peak_temperature_c']].describe().round(1))
else:
    n_workers = os.cpu_count() or 2
    print(f'Generating {N_DATASET_SAMPLES:,} samples using {n_workers} workers...')
    print('Tip: increase N_DATASET_SAMPLES in Config for better reward coverage.')

    from prodata.battery_gym.scripts.precompute_dataset import precompute_dataset
    precompute_dataset(
        n_samples  = N_DATASET_SAMPLES,
        output_path= dataset_path,
        n_workers  = n_workers,
        seed       = SEED,
    )
    df = pd.read_parquet(dataset_path)
    print(f'Done. {len(df):,} rows saved to {dataset_path}')

## 4. Load model — Unsloth 4-bit + LoRA

In [ ]:
import random
import torch
from unsloth import FastLanguageModel

# Seed everything for reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'Loading {MODEL_NAME}...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = True,
    dtype          = None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                      'gate_proj','up_proj','down_proj'],
    lora_alpha     = LORA_RANK,
    lora_dropout   = 0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device    = next(model.parameters()).device
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total params:     {total/1e6:.0f}M')
print(f'Trainable (LoRA): {trainable/1e6:.1f}M  ({trainable/total:.2%})')
print(f'VRAM used:        {torch.cuda.memory_allocated()/1e9:.1f} GB')
print(f'VRAM reserved:    {torch.cuda.memory_reserved()/1e9:.1f} GB')

## 5. Environment and reward function

In [ ]:
import json
import prodata.battery_gym
from pathlib import Path as _Path
from prodata.battery_gym.task_schema import CellTaskSpec
from prodata.battery_gym.simulators.electrochemical_sim import ElectrochemicalSimulator
from prodata.battery_gym.verifiers.basic.cell_verifier import BasicCellVerifier

# Dataset-mode simulator — KDTree lookup, ~1 ms/query
_sim_fast = ElectrochemicalSimulator(mode='dataset', dataset_path=DATASET_PATH)
_verifier = BasicCellVerifier()

# Load all 30 tasks
_tasks_path = _Path(prodata.battery_gym.__file__).parent / 'tasks' / 'cells_basic.json'
tasks       = [CellTaskSpec(**t) for t in json.loads(_tasks_path.read_text())]
task_lookup = {t.task_id: t for t in tasks}

easy_tasks   = [t for t in tasks if t.difficulty == 'easy']
medium_tasks = [t for t in tasks if t.difficulty == 'medium']
hard_tasks   = [t for t in tasks if t.difficulty == 'hard']

print(f'Tasks: {len(tasks)} total — '
      f'{len(easy_tasks)} easy / {len(medium_tasks)} medium / {len(hard_tasks)} hard')


def compute_reward(code: str, task: CellTaskSpec) -> tuple[float, dict]:
    """Score a cell design. Uses fast dataset lookup (~1 ms). Returns (reward, info)."""
    result = _sim_fast.execute(code, task.requirements.model_dump())
    verif  = _verifier.verify(result, task)
    return verif.overall_score, {
        'success':          verif.passed,
        'dimension_scores': verif.dimension_scores,
        'outputs':          result.outputs if result.success else {},
        'sim_success':      result.success,
    }


# Sanity check
_code = '''
params = {
    "chemistry": "NMC532",
    "negative_electrode_thickness": 95e-6,
    "negative_electrode_porosity": 0.28,
    "negative_electrode_particle_radius": 5e-6,
    "positive_electrode_thickness": 80e-6,
    "positive_electrode_porosity": 0.30,
    "positive_electrode_particle_radius": 3.5e-6,
    "separator_thickness": 20e-6,
    "separator_porosity": 0.45,
    "ambient_temperature_celsius": 25.0,
}
'''
_r, _info = compute_reward(_code, task_lookup['cell_ev_standard_001'])
print(f'Sanity check (good design): reward={_r:.3f}  passed={_info["success"]}')
if _r < 0.5:
    print('WARNING: dataset may be too sparse. Consider increasing N_DATASET_SAMPLES.')

## 6. Prompt format and generation

In [ ]:
SYSTEM_PROMPT = """You are an electrochemical engineer designing lithium-ion battery cells.

Generate Python code that defines `params` — a dict of cell design parameters.
Your design will be scored by PyBaMM electrochemical simulation.

Design rules (physical constraints):
  chemistry:                        "NMC532" | "LFP" | "NCA"
  negative_electrode_thickness:     20e-6 to 200e-6 m   (thicker = more capacity)
  negative_electrode_porosity:      0.10 to 0.65         (lower = denser; too low blocks Li+)
  negative_electrode_particle_radius: 0.5e-6 to 20e-6 m (smaller = better kinetics + cycle life)
  positive_electrode_thickness:     20e-6 to 180e-6 m
  positive_electrode_porosity:      0.10 to 0.65
  positive_electrode_particle_radius: 0.5e-6 to 15e-6 m
  separator_thickness:              10e-6 to 60e-6 m     (thinner = less dead weight)
  separator_porosity:               0.25 to 0.70
  ambient_temperature_celsius:      -40 to 60

Output only the Python code block. No explanation, no imports."""


def build_prompt(task: CellTaskSpec) -> str:
    req = task.requirements
    user_msg = (
        f"{task.description}\n\n"
        f"Requirements:\n"
        f"  Chemistry:        {req.chemistry}\n"
        f"  Energy density:   >= {req.target_energy_density_whkg:.0f} Wh/kg\n"
        f"  Cycle life:       >= {req.min_cycle_life} cycles to 80%\n"
        f"  Peak temperature: <= {req.max_peak_temp_c:.0f} degC\n"
        f"  Cost:             <= ${req.max_cost_kwh:.0f}/kWh\n"
        f"  Charge C-rate:    {req.c_rate_charge}C\n"
        f"  Discharge C-rate: {req.c_rate_discharge}C\n"
        f"  Ambient temp:     {req.ambient_temp_c:.0f} degC"
    )
    prompt = tokenizer.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM_PROMPT},
         {'role': 'user',   'content': user_msg}],
        tokenize=False, add_generation_prompt=True,
    )
    # Prime the output to force valid Python dict syntax
    prompt += 'params = {\n    "chemistry": '
    return prompt


def extract_params_code(raw_suffix: str) -> str:
    """
    Reconstruct the full params dict from model output.
    The model output starts after the primed prefix `params = {\n    "chemistry": `.
    Handles unclosed braces and garbage output gracefully.
    """
    code = 'params = {\n    "chemistry": ' + raw_suffix
    # Walk forward to find the closing brace
    depth, cut = 0, len(code)
    in_str, escape = False, False
    for i, ch in enumerate(code):
        if escape:
            escape = False
            continue
        if ch == '\\' and in_str:
            escape = True
            continue
        if ch == '"' and not escape:
            in_str = not in_str
            continue
        if in_str:
            continue
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                cut = i + 1
                break
    return code[:cut]


FastLanguageModel.for_inference(model)

@torch.no_grad()
def generate_code(task: CellTaskSpec, temperature: float = 0.8) -> str:
    """Generate cell design code for a task. Returns a params dict string."""
    prompt = build_prompt(task)
    inputs = tokenizer(
        prompt, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN
    ).to(device)
    try:
        out = model.generate(
            **inputs,
            max_new_tokens     = 300,
            temperature        = max(temperature, 0.01),
            do_sample          = temperature > 0.01,
            pad_token_id       = tokenizer.pad_token_id,
        )
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return 'params = {"chemistry": "NMC532"}'
    new_tokens = out[0][inputs['input_ids'].shape[-1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return extract_params_code(raw)


# Quick test
_test_task = task_lookup['cell_ev_standard_001']
_test_code = generate_code(_test_task, temperature=0.8)
_test_r, _test_info = compute_reward(_test_code, _test_task)
print('Sample generation:')
print(_test_code[:300])
print(f'\nZero-shot reward: {_test_r:.3f}  passed={_test_info["success"]}')

## 7. Zero-shot baseline

**Run this before training.** Captures codes and scores for all 30 tasks.
These exact codes are used later in the before/after comparison — we do not
re-generate after training (that would use the trained model for 'before').

In [ ]:
from tqdm.notebook import tqdm

FastLanguageModel.for_inference(model)

print('Capturing zero-shot baseline (greedy, temperature=0.1)...')
print('These codes are saved and used for the before/after comparison later.\n')

# Saved: code + score + info for every task, before any training
baseline = {}  # task_id -> {code, reward, passed, difficulty, dimension_scores, outputs}

for task in tqdm(tasks, desc='Zero-shot baseline'):
    code   = generate_code(task, temperature=0.1)
    reward, info = compute_reward(code, task)
    baseline[task.task_id] = {
        'code':             code,
        'reward':           reward,
        'passed':           info['success'],
        'difficulty':       task.difficulty,
        'dimension_scores': info['dimension_scores'],
        'outputs':          info['outputs'],
    }

print()
print(f'  {"Difficulty":8}  {"Passed":>8}  {"Avg reward":>12}')
print(f'  {"-"*35}')
for level in ('easy', 'medium', 'hard'):
    ids   = [tid for tid, v in baseline.items() if v['difficulty'] == level]
    n_pass = sum(baseline[tid]['passed'] for tid in ids)
    avg_r  = np.mean([baseline[tid]['reward'] for tid in ids])
    print(f'  {level.upper():8}  {n_pass}/{len(ids):>5}   {avg_r:>12.3f}')
n_total = sum(v['passed'] for v in baseline.values())
avg_all = np.mean([v['reward'] for v in baseline.values()])
print(f'  {"TOTAL":8}  {n_total}/30   {avg_all:>12.3f}')
print(f'\nBaseline captured. These scores are the starting line.')

## 8. RL Training — REINFORCE + LoRA

**Algorithm:** REINFORCE with running-mean advantage baseline and gradient accumulation.

Key design choices:
- **Running mean baseline** (last 30 rewards) instead of hardcoded 0.5 — adapts as the model improves
- **No skip gate** — even zero-reward episodes contribute gradient (negative advantage = model learns to avoid bad designs)
- **Gradient accumulation every 4 steps** — reduces variance of single-sample estimates
- **Temperature annealing** — 0.9 → 0.3 over training (explore early, exploit late)
- **Curriculum** — 60% easy tasks early, shifts to harder as training progresses
- **Checkpoint every 100 episodes** — safe against Colab disconnects

See Appendix for GRPO (better sample efficiency, needs more VRAM).

In [ ]:
import matplotlib.pyplot as plt
from IPython import display
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from collections import deque

# ── Checkpoint helpers ────────────────────────────────────────────────────────

def save_checkpoint(episode: int, history: dict):
    ckpt_path = Path(CHECKPOINT_DIR)
    model.save_pretrained(str(ckpt_path / 'lora_adapter'))
    tokenizer.save_pretrained(str(ckpt_path / 'lora_adapter'))
    import pickle
    with open(ckpt_path / 'history.pkl', 'wb') as f:
        pickle.dump({'episode': episode, 'history': history, 'baseline': baseline}, f)
    print(f'  [Checkpoint saved at episode {episode} → {CHECKPOINT_DIR}]')


def load_checkpoint() -> tuple[int, dict]:
    import pickle
    history_file = Path(CHECKPOINT_DIR) / 'history.pkl'
    adapter_dir  = Path(CHECKPOINT_DIR) / 'lora_adapter'
    if history_file.exists() and adapter_dir.exists():
        with open(history_file, 'rb') as f:
            saved = pickle.load(f)
        from peft import PeftModel
        global model
        model.load_adapter(str(adapter_dir), adapter_name='default')
        print(f'Resumed from checkpoint at episode {saved["episode"]}.')
        return saved['episode'] + 1, saved['history']
    return 1, {'episode': [], 'reward': [], 'passed': [],
                'energy': [], 'cycle_life': [], 'loss': []}


# ── Check for existing checkpoint ────────────────────────────────────────────
start_episode, history = load_checkpoint()
if start_episode > 1:
    print(f'Continuing from episode {start_episode} / {N_EPISODES}')
else:
    print(f'Starting fresh training: {N_EPISODES} episodes')
    history = {'episode': [], 'reward': [], 'passed': [],
               'energy': [], 'cycle_life': [], 'loss': []}

# ── Optimizer + scheduler ────────────────────────────────────────────────────
lora_params = [p for p in model.parameters() if p.requires_grad]
optimizer   = AdamW(lora_params, lr=LEARNING_RATE, weight_decay=0.01)
scheduler   = OneCycleLR(
    optimizer,
    max_lr     = LEARNING_RATE,
    total_steps= N_EPISODES // ACCUM_STEPS,
    pct_start  = 0.1,
    anneal_strategy = 'cos',
)

# ── Curriculum task sampler ───────────────────────────────────────────────────
def sample_task(episode: int) -> CellTaskSpec:
    progress = min(1.0, (episode - 1) / max(1, N_EPISODES * 0.5))
    easy_p   = EASY_START + (EASY_END - EASY_START) * progress
    r = random.random()
    if r < easy_p:
        return random.choice(easy_tasks)
    elif r < easy_p + (1.0 - easy_p) * 0.55:
        return random.choice(medium_tasks)
    else:
        return random.choice(hard_tasks)


def current_temperature(episode: int) -> float:
    progress = min(1.0, (episode - 1) / max(1, N_EPISODES - 1))
    return TEMP_START + (TEMP_END - TEMP_START) * progress


# ── Dashboard ─────────────────────────────────────────────────────────────────
def plot_dashboard(history: dict):
    display.clear_output(wait=True)
    ep = history['episode']
    if len(ep) < 5:
        return

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    W = BASELINE_WINDOW

    axes[0].plot(ep, history['reward'], alpha=0.25, color='steelblue', linewidth=0.8)
    if len(ep) >= W:
        roll = np.convolve(history['reward'], np.ones(W)/W, 'valid')
        axes[0].plot(ep[W-1:], roll, color='steelblue', linewidth=2.0, label=f'{W}-ep mean')
    axes[0].axhline(0.70, color='green', linestyle='--', alpha=0.6, label='Pass threshold')
    zs_avg = np.mean([v['reward'] for v in baseline.values()])
    axes[0].axhline(zs_avg, color='red', linestyle=':', alpha=0.5, label=f'Zero-shot ({zs_avg:.2f})')
    axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Reward')
    axes[0].set_title('Reward'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

    if len(ep) >= W:
        pass_rate = np.convolve([float(p) for p in history['passed']],
                                np.ones(W)/W, 'valid') * 100
        axes[1].plot(ep[W-1:], pass_rate, color='green', linewidth=2)
        axes[1].fill_between(ep[W-1:], 0, pass_rate, alpha=0.15, color='green')
    zs_pass = 100 * sum(v['passed'] for v in baseline.values()) / len(baseline)
    axes[1].axhline(zs_pass, color='red', linestyle=':', alpha=0.5, label=f'Zero-shot ({zs_pass:.0f}%)')
    axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Pass Rate (%)')
    axes[1].set_title(f'Pass Rate (window={W})')
    axes[1].set_ylim(0, 100); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

    valid_e = [(e, r) for e, r in zip(history['energy'], history['reward']) if e and e > 0]
    if valid_e:
        energies, rewards = zip(*valid_e[-300:])
        sc = axes[2].scatter(range(len(energies)), energies,
                             c=rewards, cmap='RdYlGn', vmin=0, vmax=1, s=12, alpha=0.7)
        plt.colorbar(sc, ax=axes[2], label='Reward')
    axes[2].set_xlabel('Recent episodes'); axes[2].set_ylabel('Energy Density (Wh/kg)')
    axes[2].set_title('Energy Density (colour = reward)'); axes[2].grid(alpha=0.3)

    plt.suptitle(f'Episode {ep[-1]}/{N_EPISODES}  |  '
                 f'Recent pass rate: {np.mean(history["passed"][-W:])*100:.0f}%  |  '
                 f'LR: {scheduler.get_last_lr()[0]:.2e}', fontsize=10)
    plt.tight_layout()
    plt.show()


# ── Training loop ─────────────────────────────────────────────────────────────
FastLanguageModel.for_training(model)

reward_window = deque(
    history['reward'][-BASELINE_WINDOW:] if history['reward'] else [],
    maxlen=BASELINE_WINDOW,
)
accum_count   = 0
accum_loss    = torch.tensor(0.0, device=device)

pbar = tqdm(range(start_episode, N_EPISODES + 1), desc='Training')

for episode in pbar:
    task        = sample_task(episode)
    temperature = current_temperature(episode)
    prompt      = build_prompt(task)

    inputs = tokenizer(
        prompt, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN
    ).to(device)

    # ── Generate ─────────────────────────────────────────────────────────────
    try:
        with torch.no_grad():
            gen_out = model.generate(
                **inputs,
                max_new_tokens = 300,
                temperature    = temperature,
                do_sample      = True,
                pad_token_id   = tokenizer.pad_token_id,
            )
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        optimizer.zero_grad()
        accum_count = 0
        accum_loss  = torch.tensor(0.0, device=device)
        pbar.write(f'  OOM at episode {episode}, cleared cache and skipped')
        continue

    # ── Decode + reward ───────────────────────────────────────────────────────
    new_tokens = gen_out[0][inputs['input_ids'].shape[-1]:]
    raw        = tokenizer.decode(new_tokens, skip_special_tokens=True)
    code       = extract_params_code(raw)
    reward, info = compute_reward(code, task)

    # ── Running mean baseline ─────────────────────────────────────────────────
    baseline_val = np.mean(reward_window) if len(reward_window) >= 5 else 0.4
    advantage    = reward - baseline_val
    reward_window.append(reward)

    # ── REINFORCE gradient step ───────────────────────────────────────────────
    # Truncate full_ids to MAX_SEQ_LEN before the forward pass.
    # gen_out can exceed MAX_SEQ_LEN (prompt + 300 new tokens); Unsloth truncates
    # input but not labels, causing a batch size mismatch without this guard.
    prompt_len = inputs['input_ids'].shape[-1]
    full_ids   = gen_out[0].unsqueeze(0)[:, :MAX_SEQ_LEN]
    labels     = full_ids.clone()
    labels[0, :prompt_len] = -100  # mask prompt tokens; only train on generated output

    with torch.enable_grad():
        out       = model(input_ids=full_ids, labels=labels)
        step_loss = out.loss * advantage / ACCUM_STEPS

    step_loss.backward()
    accum_loss  = accum_loss + step_loss.detach()
    accum_count += 1

    if accum_count >= ACCUM_STEPS:
        grad_norm = torch.nn.utils.clip_grad_norm_(lora_params, MAX_GRAD_NORM)
        if torch.isfinite(grad_norm):
            optimizer.step()
            scheduler.step()
        optimizer.zero_grad()
        history['loss'].append(accum_loss.item())
        accum_loss  = torch.tensor(0.0, device=device)
        accum_count = 0

    # ── Log ───────────────────────────────────────────────────────────────────
    out_dict = info.get('outputs', {})
    history['episode'].append(episode)
    history['reward'].append(reward)
    history['passed'].append(info['success'])
    history['energy'].append(out_dict.get('energy_density_whkg', 0.0))
    history['cycle_life'].append(out_dict.get('cycle_life_80pct', 0))

    pbar.set_postfix(
        task    = task.task_id[-18:],
        r       = f'{reward:.2f}',
        adv     = f'{advantage:+.2f}',
        temp    = f'{temperature:.2f}',
        pass_30 = f'{np.mean(list(history["passed"])[-30:])*100:.0f}%',
    )

    if episode % CHECKPOINT_EVERY == 0:
        save_checkpoint(episode, history)

    if episode % 20 == 0:
        torch.cuda.empty_cache()

    if episode % VIZ_EVERY == 0:
        plot_dashboard(history)
        used  = torch.cuda.memory_allocated() / 1e9
        total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        pbar.write(f'  ep {episode:4d}  VRAM {used:.1f}/{total_vram:.1f} GB  '
                   f'LR={scheduler.get_last_lr()[0]:.2e}  '
                   f'temp={temperature:.2f}')

# Final checkpoint
save_checkpoint(N_EPISODES, history)
print(f'\nTraining complete. Final pass rate: {np.mean(history["passed"][-50:])*100:.0f}% (last 50 eps)')

## 9. Post-training evaluation

In [ ]:
FastLanguageModel.for_inference(model)

print('Post-training evaluation (greedy, temperature=0.1)...\n')
post = {}

for task in tqdm(tasks, desc='Post-eval'):
    code = generate_code(task, temperature=0.1)
    reward, info = compute_reward(code, task)
    post[task.task_id] = {
        'code':             code,
        'reward':           reward,
        'passed':           info['success'],
        'difficulty':       task.difficulty,
        'dimension_scores': info['dimension_scores'],
        'outputs':          info['outputs'],
    }

# ── Comparison table ──────────────────────────────────────────────────────────
W = 50
print(f"{'='*68}")
print(f"  {'Metric':30} {'Before':>10} {'After':>10} {'Delta':>8}")
print(f"  {'-'*64}")

for level in ('easy', 'medium', 'hard'):
    ids = [t.task_id for t in tasks if t.difficulty == level]
    b_pass = sum(baseline[i]['passed'] for i in ids) / len(ids)
    a_pass = sum(post[i]['passed']     for i in ids) / len(ids)
    b_avg  = np.mean([baseline[i]['reward'] for i in ids])
    a_avg  = np.mean([post[i]['reward']     for i in ids])
    print(f"  {level.upper()+' pass rate':30} {b_pass:>9.1%} {a_pass:>9.1%} {a_pass-b_pass:>+8.1%}")
    print(f"  {level.upper()+' avg reward':30} {b_avg:>10.3f} {a_avg:>10.3f} {a_avg-b_avg:>+8.3f}")

b_total_pass = sum(v['passed'] for v in baseline.values()) / 30
a_total_pass = sum(v['passed'] for v in post.values())     / 30
b_total_avg  = np.mean([v['reward'] for v in baseline.values()])
a_total_avg  = np.mean([v['reward'] for v in post.values()])
print(f"  {'-'*64}")
print(f"  {'TOTAL pass rate':30} {b_total_pass:>9.1%} {a_total_pass:>9.1%} {a_total_pass-b_total_pass:>+8.1%}")
print(f"  {'TOTAL avg reward':30} {b_total_avg:>10.3f} {a_total_avg:>10.3f} {a_total_avg-b_total_avg:>+8.3f}")
print(f"{'='*68}")

## 10. Before / After — charts

Uses the zero-shot codes captured in Step 7 (before training) and the
post-training codes from Step 9. Both are run through live PyBaMM for
the discharge curves and degradation plots.

In [ ]:
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')   # Colab inline rendering
import matplotlib.pyplot as plt

from prodata.battery_gym.viz import (
    plot_before_after, plot_score_distribution, plot_pareto
)
from prodata.battery_gym.simulators.electrochemical_sim import ElectrochemicalSimulator

plt.rcParams['figure.dpi'] = 110

# ── Score distribution shift ──────────────────────────────────────────────────
before_scores = [baseline[t.task_id]['reward'] for t in tasks]
after_scores  = [post[t.task_id]['reward']     for t in tasks]

fig = plot_score_distribution(before_scores, after_scores)
plt.title('Score Distribution: Before vs After RL Training')
plt.show()

# ── Pareto: energy density vs cycle life ──────────────────────────────────────
before_outputs = [baseline[t.task_id]['outputs'] for t in tasks]
after_outputs  = [post[t.task_id]['outputs']     for t in tasks]

fig = plot_pareto(
    before_results     = before_outputs,
    after_results      = after_outputs,
    task_target_energy = 250.0,
    task_target_cycles = 800,
)
plt.show()

# ── Full 2x2 panel: most improved task ───────────────────────────────────────
# Find task with largest reward improvement
improvements = {tid: post[tid]['reward'] - baseline[tid]['reward']
                for tid in baseline if tid in post}
best_tid  = max(improvements, key=improvements.get)
best_task = task_lookup[best_tid]

print(f'Most improved task: {best_tid}')
print(f'  Before: {baseline[best_tid]["reward"]:.3f}  '
      f'After: {post[best_tid]["reward"]:.3f}  '
      f'(+{improvements[best_tid]:.3f})')

# Run live PyBaMM on the SAVED codes — not re-generated.
# before = zero-shot code captured pre-training (Step 7)
# after  = post-training code from Step 9
# Neither is re-generated here — this is the real before/after.
print('\nRunning live PyBaMM for discharge curves (2 sims, ~10 s)...')
_live_sim  = ElectrochemicalSimulator(mode='live')
_task_spec = best_task.requirements.model_dump()

result_before = _live_sim.execute(baseline[best_tid]['code'], _task_spec)
result_after  = _live_sim.execute(post[best_tid]['code'],     _task_spec)

fig = plot_before_after(
    before_sim_outputs = result_before.outputs,
    after_sim_outputs  = result_after.outputs,
    before_scores      = baseline[best_tid]['dimension_scores'],
    after_scores       = post[best_tid]['dimension_scores'],
    task_description   = best_task.description[:120],
    save_path          = f'before_after_{best_tid}.png',
)
plt.show()
print(f'Figure saved -> before_after_{best_tid}.png')

## 11. Save LoRA adapter

In [ ]:
ADAPTER_PATH = Path(CHECKPOINT_DIR) / 'lora_adapter_final'
model.save_pretrained(str(ADAPTER_PATH))
tokenizer.save_pretrained(str(ADAPTER_PATH))
print(f'LoRA adapter saved → {ADAPTER_PATH}')
print(f'Size: ~{sum(f.stat().st_size for f in ADAPTER_PATH.rglob("*") if f.is_file()) / 1e6:.0f} MB')
print()
print('To reload later:')
print('  from unsloth import FastLanguageModel')
print(f'  model, tokenizer = FastLanguageModel.from_pretrained("{ADAPTER_PATH}")')
print()
print('To export as merged fp16 (for deployment):')
print(f'  model.save_pretrained_merged("battery_merged", tokenizer, save_method="merged_16bit")')

---
## Appendix: Upgrade to GRPO

GRPO samples **G=4 completions per prompt**, computes group-relative advantages, and trains on all of them together.
This is 4x more sample-efficient than REINFORCE (DeepSeek-R1 / Qwen3 use this exact approach).

**VRAM cost:** G=4 means 4x the activation memory per step. On a T4 (16 GB) you may hit OOM
with 7B. Use 3B model (`Qwen2.5-Coder-3B-Instruct-bnb-4bit`) for GRPO on T4.

**When to use:** If you have an A100 (40 GB+) or are using the 1.5B/3B model on T4.
For 7B on T4, stick with the REINFORCE loop above.

In [ ]:
# Run this cell INSTEAD of the REINFORCE loop (cell 8)
# Requires trl >= 0.12  and  3B model on T4  (or 7B on A100)

from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

# Build HuggingFace dataset with task_id as a column
# TRL will pass any extra dataset columns to the reward function as kwargs
hf_data = Dataset.from_list([
    {'prompt': build_prompt(t), 'task_id': t.task_id}
    for t in tasks
])


def battery_reward_fn(
    completions: list[str],
    task_id:     list[str],
    **kwargs,
) -> list[float]:
    """
    Reward function for GRPOTrainer.

    TRL calls this once per prompt group:
      completions = [G completion strings for one prompt]
      task_id     = [same task_id repeated G times]

    Returns G float rewards in [0, 1].
    """
    rewards = []
    for code_suffix, tid in zip(completions, task_id):
        code   = extract_params_code(code_suffix)
        task   = task_lookup.get(tid, tasks[0])
        reward, _ = compute_reward(code, task)
        rewards.append(float(reward))
    return rewards


grpo_config = GRPOConfig(
    output_dir                  = str(Path(CHECKPOINT_DIR) / 'grpo'),
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = ACCUM_STEPS,
    num_generations             = 4,         # G: completions per prompt
    max_new_tokens              = 300,
    max_prompt_length           = MAX_SEQ_LEN - 300,
    learning_rate               = LEARNING_RATE,
    num_train_epochs            = max(1, N_EPISODES // len(tasks)),
    logging_steps               = 10,
    save_steps                  = CHECKPOINT_EVERY // len(tasks),
    warmup_steps                = 20,
    fp16                        = True,
    report_to                   = 'none',
    dataloader_num_workers      = 0,         # avoid multiprocessing + PyBaMM issues
    temperature                 = TEMP_START,
)

FastLanguageModel.for_training(model)

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [battery_reward_fn],
    args             = grpo_config,
    train_dataset    = hf_data,
)

trainer.train()
print('GRPO training complete.')